# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [8]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [9]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [10]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [11]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [12]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [13]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [14]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

In [15]:
patent_states = patents.select(
    col("PATENT").alias("PATENT_ID"),
    col("POSTATE").alias("STATE")
)

patent_states.show(5)


+---------+-----+
|PATENT_ID|STATE|
+---------+-----+
|  3070801| NULL|
|  3070802|   TX|
|  3070803|   IL|
|  3070804|   OH|
|  3070805|   CA|
+---------+-----+
only showing top 5 rows



In [16]:
cited_states = citations.alias("c").join(
    patent_states.alias("p"),
    col("c.CITED") == col("p.PATENT_ID"),
    "left"
).select(
    col("c.CITING"),
    col("c.CITED"),
    col("p.STATE").alias("CITED_STATE")
)

cited_states.show(10)

+-------+-------+-----------+
| CITING|  CITED|CITED_STATE|
+-------+-------+-----------+
|3858242|1515701|       NULL|
|3858242|3319261|         OH|
|3858241|3634889|         OH|
|3858241| 956203|       NULL|
|3858241|1324234|       NULL|
|3858243|2949611|       NULL|
|3858243|3146465|         MI|
|3858241|3398406|         FL|
|3858241|3557384|         MA|
|3858242|3668705|         WI|
+-------+-------+-----------+
only showing top 10 rows



In [17]:
citation_states = cited_states.alias("c").join(
    patent_states.alias("p"),
    col("c.CITING") == col("p.PATENT_ID"),
    "left"
).select(
    col("c.CITED"),
    col("c.CITED_STATE"),
    col("c.CITING"),
    col("p.STATE").alias("CITING_STATE")
)

citation_states = citation_states.cache()

citation_states.show(10)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|1331793|       NULL|3858258|          CA|
|1540798|       NULL|3858258|          CA|
| 924225|       NULL|3858527|        NULL|
|3638586|         CA|3858527|        NULL|
|2444326|       NULL|3858527|        NULL|
|3699902|         OH|3858527|        NULL|
|2705120|       NULL|3858527|        NULL|
|2967080|       NULL|3858527|        NULL|
|3602157|         TX|3858527|        NULL|
| 957631|       NULL|3858560|          IN|
+-------+-----------+-------+------------+
only showing top 10 rows



In [18]:
citation_states.filter(
    col("CITING") == 6009554
).show(20, truncate=False)

+-------+-----------+-------+------------+
|CITED  |CITED_STATE|CITING |CITING_STATE|
+-------+-----------+-------+------------+
|4029274|NY         |6009554|NY          |
|4181849|NY         |6009554|NY          |
|4494717|NULL       |6009554|NY          |
|4831521|NY         |6009554|NY          |
|5048064|NY         |6009554|NY          |
|4611291|NY         |6009554|NY          |
|4617662|NY         |6009554|NY          |
|4740972|NY         |6009554|NY          |
|5364047|NY         |6009554|NY          |
+-------+-----------+-------+------------+



In [19]:
same_state = citation_states.filter(
    col("CITED_STATE").isNotNull() &
    col("CITING_STATE").isNotNull() &
    (col("CITED_STATE") == col("CITING_STATE"))
)

same_state.show(10)

+-------+-----------+-------+------------+
|  CITED|CITED_STATE| CITING|CITING_STATE|
+-------+-----------+-------+------------+
|3368197|         MI|3859627|          MI|
|3722929|         CA|3860191|          CA|
|3172282|         AZ|3861180|          AZ|
|3791450|         MA|3861473|          MA|
|3802510|         MA|3861473|          MA|
|3118651|         MI|3862577|          MI|
|3099569|         PA|3862844|          PA|
|3769543|         NY|3863090|          NY|
|3467396|         MI|3863935|          MI|
|3167490|         NY|3864160|          NY|
+-------+-----------+-------+------------+
only showing top 10 rows



In [20]:
same_state_counts = same_state.groupBy("CITING").agg(
    count("*").alias("SAME_STATE")
)

same_state_counts.show(10)

+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|3859627|         1|
|3860191|         1|
|3861180|         1|
|3861473|         2|
|3862577|         1|
|3862844|         1|
|3863090|         1|
|3863935|         1|
|3864160|         1|
|3864244|         2|
+-------+----------+
only showing top 10 rows



In [21]:
same_state_counts.filter(
    col("CITING") == 6009554
).show()

+-------+----------+
| CITING|SAME_STATE|
+-------+----------+
|6009554|         8|
+-------+----------+



In [22]:
augmented_patents = patents.alias("p").join(
    same_state_counts.alias("s"),
    col("p.PATENT") == col("s.CITING"),
    "left"
).drop("CITING")

In [23]:
augmented_patents = augmented_patents.fillna({
    "SAME_STATE": 0
})

In [24]:
augmented_patents.select(
    "PATENT",
    "POSTATE",
    "SAME_STATE"
).show(10)

+-------+-------+----------+
| PATENT|POSTATE|SAME_STATE|
+-------+-------+----------+
|3070803|     IL|         0|
|3070805|     CA|         0|
|3070811|     CA|         0|
|3070807|     OH|         0|
|3070801|   NULL|         0|
|3070809|     AZ|         0|
|3070810|     IL|         0|
|3070804|     OH|         0|
|3070808|     IA|         0|
|3070802|     TX|         0|
+-------+-------+----------+
only showing top 10 rows



In [25]:
top10_dataframe = augmented_patents.orderBy(
    col("SAME_STATE").desc()
).limit(10)

top10_dataframe.show(truncate=False)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|PATENT |GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466|1999 |14515|1997   |US     |CA     |5310    |2      |NULL  |326   |4  |46    |159  |0       |1.0     |NULL   |0.6186  |NULL    |4.8868  |0.0455  |0.044   |NULL    |NULL    |125       |
|5983822|1999 |14564|1998   |US     |TX     |569900  |2      |NULL  |114   |5  |55    |200  |0       |0.995   |NULL   |0.7201  |NULL    |12.45   |0.0     |0.0     |NULL    |NULL    |103       |
|6008204|1999 |14606|1998   |U